# Cat and Dog Image Classifier
A complete Google Colab notebook for the freeCodeCamp TensorFlow/Keras challenge. Run cells from top to bottom.

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
# Download and extract the challenge dataset
URL = 'https://cdn.freecodecamp.org/project-data/cats-and-dogs/cats_and_dogs.zip'
zip_path = tf.keras.utils.get_file('cats_and_dogs.zip', origin=URL, extract=True)
PATH = os.path.join(os.path.dirname(zip_path), 'cats_and_dogs')
train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')
test_dir = os.path.join(PATH, 'test')

IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 100
total_train = 2000
total_val = 1000
total_test = 50

print('Train directory:', train_dir)
print('Validation directory:', validation_dir)
print('Test directory:', test_dir)


## Cell 3 — Create image generators

In [ ]:
train_image_generator = ImageDataGenerator(rescale=1./255)
validation_image_generator = ImageDataGenerator(rescale=1./255)
test_image_generator = ImageDataGenerator(rescale=1./255)

train_data_gen = train_image_generator.flow_from_directory(
    directory=train_dir,
    batch_size=BATCH_SIZE,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)
val_data_gen = validation_image_generator.flow_from_directory(
    directory=validation_dir,
    batch_size=BATCH_SIZE,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)
test_data_gen = test_image_generator.flow_from_directory(
    directory=PATH,
    classes=['test'],
    batch_size=BATCH_SIZE,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None,
    shuffle=False
)


## Cell 4 — Plot sample images

In [ ]:
def plotImages(images_arr, probabilities=False):
    fig, axes = plt.subplots(1, len(images_arr), figsize=(20, 20))
    axes = np.array(axes).reshape(-1)
    for i, (img, ax) in enumerate(zip(images_arr, axes)):
        ax.imshow(img)
        ax.axis('off')
        if probabilities is not False:
            label = 'Dog' if probabilities[i] == 1 else 'Cat'
            confidence = probabilities[i] if isinstance(probabilities[i], float) else None
            ax.set_title(label)
    plt.tight_layout()
    plt.show()

sample_training_images, _ = next(train_data_gen)
plotImages(sample_training_images[:5])


## Cell 5 — Add data augmentation to reduce overfitting

In [ ]:
train_image_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_data_gen = train_image_generator.flow_from_directory(
    directory=train_dir,
    batch_size=BATCH_SIZE,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)


## Cell 6 — View augmented versions of one image

In [ ]:
augmented_images, _ = next(train_data_gen)
plotImages([augmented_images[0] for _ in range(5)])


## Cell 7 — Build and compile the CNN

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## Cell 8 — Train the model

In [ ]:
EPOCHS = 20
history = model.fit(
    x=train_data_gen,
    steps_per_epoch=max(1, total_train // BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=val_data_gen,
    validation_steps=max(1, total_val // BATCH_SIZE)
)


## Cell 9 — Plot training and validation accuracy/loss

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()


## Cell 10 — Predict the 50 test images

In [ ]:
test_data_gen.reset()
raw_predictions = model.predict(test_data_gen, steps=int(np.ceil(total_test / BATCH_SIZE)))
probabilities = [1 if prediction[0] >= 0.5 else 0 for prediction in raw_predictions[:total_test]]
test_images = next(test_image_generator.flow_from_directory(
    directory=PATH,
    classes=['test'],
    batch_size=total_test,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None,
    shuffle=False
))
plotImages(test_images[:5], probabilities[:5])
print('Predictions generated:', len(probabilities))


## Cell 11 — Basic challenge check
The original freeCodeCamp notebook may provide its own `test` function and grading assertions. If so, run that supplied cell as well.

In [ ]:
# Report validation accuracy. The challenge target is at least 63%.
best_val_accuracy = max(history.history['val_accuracy'])
print(f'Best validation accuracy: {best_val_accuracy:.2%}')
if best_val_accuracy >= 0.63:
    print('Validation accuracy target reached.')
else:
    print('Target not reached yet. Try more epochs or tune augmentation/model settings.')
